# Ensemble forecast evaluation

This notebook shows how to use `ForecastPerformance` to evaluate **ensemble forecasts** with a suite of skill metrics.

In [ ]:
import pandas as pd
from pathlib import Path
from performance import ForecastPerformance, Results 

test_dataset_path = Path(r'tests\test_datasets')

obs = pd.read_csv(test_dataset_path / 'observed.csv', sep=';', index_col=0, parse_dates=True)
obs.columns = pd.Index(pd.to_timedelta(obs.columns), name='leadtime')
obs.index.name = 'event_datetime'

ens_path = test_dataset_path / 'ens.parquet'
ens_parquet = pd.read_parquet(ens_path)
display(ens_parquet.head(5))

In [ ]:
mean = ens_parquet.unstack('ensemble_member').mean(axis=1)
median = ens_parquet.unstack('ensemble_member').median(axis=1)

results = Results('Model', 'Metric', 'Leadtime')

fp = ForecastPerformance(obs)
fp.add_by_production_date(mean.droplevel('event_datetime').unstack('leadtime'), name='mean')
fp.add_by_production_date(median.droplevel('event_datetime').unstack('leadtime'), name='median')

for model in fp.names():
    print(f'{model}')
    for metric in [fp.KGEprime, fp.NSE, fp.RMSE, fp.MAE, fp.bias, fp.relative_bias, fp.Pearson]:
        for leadtime in median.index.get_level_values('leadtime').unique():
            results.append(Model=model, Metric=metric.__name__, Leadtime=leadtime,
                        Value=fp.deterministic(metric, model, leadtime=leadtime),
                        )
        
df = results.to_pandas(index=['Metric', 'Model'], columns=['Leadtime'])
display(df.iloc[:, ::6])

In [ ]:
results = Results('Model', 'Metric', 'Leadtime')

fp = ForecastPerformance(obs)
fp.add_by_production_date(ens_parquet.droplevel('event_datetime').unstack(['leadtime', 'ensemble_member']), name='ens')

for model in fp.names():
    print(f'{model}')
    for metric in [fp.reliability, fp.resolution_relative, fp.resolution]:
        for leadtime in ens_parquet.index.get_level_values('leadtime').unique():
            results.append(Model=model, Metric=metric.__name__, Leadtime=leadtime,
                        Value=metric(model, leadtimes=[leadtime]),
                        )
            
for model in fp.names():
    print(f'{model}')
    for metric in [fp.CRPS, fp.fairCRPS]:
        for leadtime in ens_parquet.index.get_level_values('leadtime').unique():
            results.append(Model=model, Metric=metric.__name__, Leadtime=leadtime,
                        Value=metric(model, leadtime=leadtime),
                        )

df = results.to_pandas(index=['Metric', 'Model'], columns=['Leadtime'])
display(df.iloc[:, ::6])

In [ ]:
fp.QQPlot(name='ens', leadtimes=ens_parquet.index.get_level_values('leadtime').unique()[::6])